# Reclamações 04 · Análise

## 0. Contexto e leitura dos números

Este notebook responde às cinco perguntas de negócio sobre reclamações, na ordem de uma narrativa: do que o cliente reclama, quem reduziu as reclamações, se a procedência conta toda a história, se técnico e comercial andam juntos e se as reclamações técnicas confirmam o ranking de continuidade. Cada seção abre com a pergunta, mostra as visões que a respondem e fecha com a leitura dos números.

| # | Pergunta | Seção |
|---|---|---|
| P1 | Quais grupos de reclamação comercial mais pesam para o consumidor e quais mais contribuíram para a variação? | 1 e 2 |
| P2 | Quais distribuidoras mais reduziram as reclamações comerciais recebidas e procedentes por mil UCs, e em quais grupos essa redução ocorreu? | 2 |
| P3 | A redução das procedentes, se houve, vem acompanhada de redução das recebidas, ou pode refletir maior rigor na classificação de procedência? | 3 |
| P4 | As reclamações técnicas e comerciais evoluem juntas em cada distribuidora? | 4 |
| P5 | O ranking de reclamações de Qualidade confirma o ranking de continuidade (DEC-FI e FEC-FI) na mesma janela? | 5 |

### Como ler os números

- **Universo:** as distribuidoras de grande porte, sem a CELESC, excluída por série internamente inconsistente (`02_silver_complaints`, seção 11).
- **Nível:** só o nível 1. A ouvidoria recebe reclamações que já passaram pelo nível 1, e somar os dois contaria o mesmo problema duas vezes.
- **Indicador:** reclamações em 12 meses por mil unidades consumidoras (UCs), com o `NumCon` da base de continuidade no denominador.
- **Comparação:** janela de 12 meses encerrada em dezembro de 2024 contra a encerrada em junho de 2026, com pontos intermediários em junho e dezembro de 2025.
- **Posição:** 1 é a maior redução.

### Limites declarados

1. O indicador segue a regra de exclusão do FER (PRODIST Módulo 8, item 285), mas não é o FER oficial: o denominador é a média mensal de UCs na janela, e não o número de consumidores de dezembro. Os valores não são comparáveis a FER publicados.
2. O recorte comercial estrito é decisão deste trabalho: retira do item 285 Rede/Manutenção e Outros de Qualidade, que fariam o indicador medir causa técnica.
3. O grupo Qualidade inclui interrupção programada, tensão e outros de qualidade, 2,4% do grupo em 2024.
4. Pagamento e Outras comerciais têm volume baixo por distribuidora; a variação percentual é mais sensível a oscilação, e o volume absoluto acompanha a posição.
5. Os resultados são descritivos. Só as correlações das P4 e P5 trazem valor-p; a inferência sobre a série mensal fica como trabalho futuro.

## Configuração

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import functions as F
from scipy.stats import spearmanr

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_GOLD, SCHEMA_SILVER

SILVER = f"{CATALOG}.{SCHEMA_SILVER}"
GOLD = f"{CATALOG}.{SCHEMA_GOLD}"

RECORTE_TOTAL = "comercial_estrito"
GRUPOS_COMERCIAIS = ["faturamento", "pagamento", "outras_comerciais"]
ROTULO_GRUPO = {"faturamento": "Faturamento", "pagamento": "Pagamento",
                "outras_comerciais": "Outras comerciais", "qualidade": "Qualidade"}
COR_GRUPO = {"faturamento": "#1f4e79", "pagamento": "#c9a227", "outras_comerciais": "#8faadc"}
COR_MELHORA, COR_PIORA, COR_NEUTRA = "#2e7d32", "#c62828", "#607d8b"
MESES = {1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun",
         7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez"}

sns.set_theme(style="whitegrid", context="notebook")


def rotulo(ano_mes):
    """Short Portuguese label for a yyyymm window end, e.g. 202412 -> dez/24."""
    return f"{MESES[ano_mes % 100]}/{str(ano_mes // 100)[2:]}"


nomes = (spark.table(f"{SILVER}.dim_distribuidora")
         .select("num_cnpj", F.trim("sig_agente").alias("sig_agente")).toPandas())

janela = spark.table(f"{GOLD}.fato_reclamacao_janela").toPandas().merge(nomes, on="num_cnpj")
ranking = spark.table(f"{GOLD}.ranking_reclamacoes").toPandas().merge(nomes, on="num_cnpj")
ranking_cont = spark.table(f"{GOLD}.ranking_continuidade").toPandas().merge(nomes, on="num_cnpj")

FINS = sorted(janela["fim_janela"].unique())
INI, FIM = FINS[0], FINS[-1]
univ = janela[~janela["excluida_ranking"] & janela["janela_valida"]].copy()


def rk(recorte, medida):
    """One ranking as a pandas frame indexed by company."""
    return (ranking[(ranking["recorte"] == recorte) & (ranking["medida"] == medida)]
            .set_index("sig_agente").sort_values("posicao"))


def valor(recorte, fim, coluna):
    """One indicator of one window, indexed by company."""
    return univ[(univ["recorte"] == recorte) & (univ["fim_janela"] == fim)].set_index("sig_agente")[coluna]


print(f"Janelas.........: {[rotulo(f) for f in FINS]}")
print(f"Comparacao......: {rotulo(INI)} contra {rotulo(FIM)}")
print(f"Distribuidoras..: {univ['num_cnpj'].nunique()} no universo analisado")

## 1. Do que o cliente reclama (P1, primeira parte)

Antes de perguntar quem melhorou, é preciso saber do que o cliente reclama. A primeira visão mostra o peso de cada grupo da REH 2.992/2021 nas reclamações procedentes do nível 1, na última janela.

In [ ]:
fato = spark.table(f"{SILVER}.fato_manifestacao")
tipologia = spark.table(f"{SILVER}.dim_tipologia")
universo_spark = spark.createDataFrame(univ[["num_cnpj"]].drop_duplicates())
meses_fim = [m for m in range(FIM - 100 + 1, FIM + 1) if 1 <= m % 100 <= 12] if FIM % 100 == 12 else \
    [(FIM // 100 - 1) * 100 + m for m in range(FIM % 100 + 1, 13)] + [(FIM // 100) * 100 + m for m in range(1, FIM % 100 + 1)]

familia = (fato.filter((F.col("nivel") == 1) & F.col("ano_mes").isin(meses_fim))
    .join(universo_spark, "num_cnpj")
    .join(tipologia.filter((F.col("cod_nivel_1") == "102") & ~F.col("ind_subtotal")), "cod_tipologia")
    .groupBy("cod_nivel_2", "desc_nivel_2")
    .agg(F.sum("qtd_procedentes").alias("procedentes"))
    .toPandas())
familia["participacao"] = familia["procedentes"] / familia["procedentes"].sum() * 100
familia = familia.sort_values("participacao", ascending=False)

principais = familia.head(4).copy()
demais = pd.DataFrame([{"desc_nivel_2": "Demais grupos",
                        "participacao": familia["participacao"].iloc[4:].sum()}])
grafico = pd.concat([principais, demais])

fig, ax = plt.subplots(figsize=(8, 3.2))
sns.barplot(data=grafico, y="desc_nivel_2", x="participacao", color=COR_NEUTRA, ax=ax)
for i, v in enumerate(grafico["participacao"]):
    ax.text(v + 0.8, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Participação nas procedentes da família 102 (%)")
ax.set_ylabel("")
ax.set_xlim(0, 105)
ax.set_title(f"Procedentes do nível 1 por grupo da REH, janela encerrada em {rotulo(FIM)}")
plt.tight_layout()
plt.show()

print(f"procedentes da familia 102 na janela: {familia['procedentes'].sum():,.0f}")

#### Composição do comercial estrito:

Separado o técnico, a pergunta passa a ser o peso de cada grupo dentro do comercial estrito, e se ele muda ao longo das janelas.

In [ ]:
comp = (univ[univ["recorte"].isin(GRUPOS_COMERCIAIS)]
        .groupby(["fim_janela", "recorte"])[["qtd_procedentes", "qtd_recebidas"]].sum()
        .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, medida, titulo in zip(axes, ["qtd_procedentes", "qtd_recebidas"], ["Procedentes", "Recebidas"]):
    tabela = comp.pivot(index="fim_janela", columns="recorte", values=medida)[GRUPOS_COMERCIAIS]
    tabela = tabela.div(tabela.sum(axis=1), axis=0) * 100
    base = np.zeros(len(tabela))
    for grupo in GRUPOS_COMERCIAIS:
        ax.bar([rotulo(f) for f in tabela.index], tabela[grupo], bottom=base,
               color=COR_GRUPO[grupo], label=ROTULO_GRUPO[grupo])
        for x, (b, v) in enumerate(zip(base, tabela[grupo])):
            ax.text(x, b + v / 2, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9)
        base += tabela[grupo].values
    ax.set_title(titulo)
    ax.set_ylabel("Participação no comercial estrito (%)")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
fig.suptitle("Composição do comercial estrito por janela de 12 meses")
plt.tight_layout()
plt.show()

participacao = comp.pivot(index="fim_janela", columns="recorte", values="qtd_procedentes")[GRUPOS_COMERCIAIS]
display((participacao.div(participacao.sum(axis=1), axis=0) * 100).round(1)
        .rename(index=rotulo, columns=ROTULO_GRUPO).reset_index())

#### Composição por distribuidora:

O agregado pode esconder distribuidoras em que outro grupo lidera. A visão abaixo mostra a composição das procedentes do comercial estrito em cada distribuidora, nas janelas inicial e final.

In [ ]:
por_dx = univ[univ["recorte"].isin(GRUPOS_COMERCIAIS) & univ["fim_janela"].isin([INI, FIM])]
por_dx = por_dx.pivot_table(index=["sig_agente", "fim_janela"], columns="recorte",
                            values="qtd_procedentes", aggfunc="sum")[GRUPOS_COMERCIAIS]
por_dx = por_dx.div(por_dx.sum(axis=1), axis=0) * 100
ordem = por_dx.xs(FIM, level="fim_janela").sort_values("faturamento").index

fig, axes = plt.subplots(1, 2, figsize=(12, max(6, len(ordem) * 0.28)), sharey=True)
for ax, fim in zip(axes, [INI, FIM]):
    tabela = por_dx.xs(fim, level="fim_janela").reindex(ordem)
    base = np.zeros(len(tabela))
    for grupo in GRUPOS_COMERCIAIS:
        ax.barh(tabela.index, tabela[grupo], left=base, color=COR_GRUPO[grupo], label=ROTULO_GRUPO[grupo])
        base += tabela[grupo].fillna(0).values
    ax.set_title(f"Janela encerrada em {rotulo(fim)}")
    ax.set_xlabel("Participação nas procedentes (%)")
    ax.tick_params(axis="y", labelsize=8)
axes[1].legend(loc="lower right", frameon=True, fontsize=8)
fig.suptitle("Composição das procedentes do comercial estrito por distribuidora")
plt.tight_layout()
plt.show()

lider = por_dx.xs(FIM, level="fim_janela").idxmax(axis=1)
print("grupo lider por distribuidora na ultima janela:")
print(lider.map(ROTULO_GRUPO).value_counts().to_string())
print("\ndistribuidoras em que o Faturamento nao lidera:", sorted(lider[lider != "faturamento"].index))

#### Leitura:

LEITURA_1

## 2. Quem reduziu as reclamações comerciais (P2 e P1, segunda parte)

O ranking do comercial estrito total responde diretamente à P2: para o consumidor, o tipo da reclamação é indiferente, e o total mede o volume de problemas comerciais que ele enfrenta. Os rankings por grupo e a decomposição mostram de onde veio a variação.

In [ ]:
total = rk(RECORTE_TOTAL, "procedentes")

fig, ax = plt.subplots(figsize=(9, max(5, len(total) * 0.28)))
cores = [COR_MELHORA if v < 0 else COR_PIORA for v in total["var_pct"]]
ax.barh(total.index[::-1], total["var_pct"][::-1], color=cores[::-1])
ax.axvline(0, color="#37474f", linewidth=1)
ax.set_xlabel(f"Variação das procedentes por mil UCs, {rotulo(INI)} a {rotulo(FIM)} (%)")
ax.tick_params(axis="y", labelsize=8)
ax.set_title("P2 - Comercial estrito: variação das procedentes por mil UCs")
plt.tight_layout()
plt.show()

display(total[["posicao", "indicador_inicio", "indicador_fim", "var_pct", "var_abs",
               "volume_inicio", "volume_fim"]].reset_index())

#### Trajetória nas quatro janelas:

A variação entre a primeira e a última janela não diz se a melhora foi contínua ou concentrada. As linhas mostram o indicador nas quatro janelas, com base 100 na primeira, para as cinco melhores e as cinco piores posições.

In [ ]:
serie = (univ[univ["recorte"] == RECORTE_TOTAL]
         .pivot(index="sig_agente", columns="fim_janela", values="procedentes_por_mil"))
indice = serie.div(serie[INI], axis=0) * 100
melhores, piores = total.index[:5], total.index[-5:]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, grupo, titulo in zip(axes, [melhores, piores], ["Cinco maiores reduções", "Cinco maiores aumentos"]):
    for dx in grupo:
        ax.plot([rotulo(f) for f in FINS], indice.loc[dx, FINS], marker="o", label=dx)
    ax.axhline(100, color="#37474f", linewidth=0.8, linestyle="--")
    ax.set_title(titulo)
    ax.set_ylabel(f"Índice (base {rotulo(INI)} = 100)")
    ax.legend(fontsize=8, frameon=False)
fig.suptitle("Trajetória das procedentes por mil UCs no comercial estrito")
plt.tight_layout()
plt.show()

#### Decomposição da variação por grupo:

Como os grupos comerciais somam o total e usam o mesmo denominador, a variação do indicador total é a soma das variações dos grupos. O gráfico mostra quanto cada grupo contribuiu, em procedentes por mil UCs, para a variação de cada distribuidora.

In [ ]:
delta = pd.DataFrame({g: valor(g, FIM, "procedentes_por_mil") - valor(g, INI, "procedentes_por_mil")
                      for g in GRUPOS_COMERCIAIS})
delta["total"] = valor(RECORTE_TOTAL, FIM, "procedentes_por_mil") - valor(RECORTE_TOTAL, INI, "procedentes_por_mil")
delta = delta.loc[total.index].sort_values("total")

residuo = (delta[GRUPOS_COMERCIAIS].sum(axis=1) - delta["total"]).abs().max()
print(f"maior diferenca entre a soma dos grupos e o total: {residuo:.4f} procedentes por mil UCs")

fig, ax = plt.subplots(figsize=(9, max(5, len(delta) * 0.28)))
pos_base = np.zeros(len(delta))
neg_base = np.zeros(len(delta))
for grupo in GRUPOS_COMERCIAIS:
    v = delta[grupo].values
    left = np.where(v >= 0, pos_base, neg_base)
    ax.barh(delta.index, v, left=left, color=COR_GRUPO[grupo], label=ROTULO_GRUPO[grupo])
    pos_base += np.where(v >= 0, v, 0)
    neg_base += np.where(v < 0, v, 0)
ax.scatter(delta["total"], delta.index, color="black", s=14, zorder=3, label="Variação total")
ax.axvline(0, color="#37474f", linewidth=1)
ax.set_xlabel("Contribuição para a variação (procedentes por mil UCs)")
ax.tick_params(axis="y", labelsize=8)
ax.legend(fontsize=8, frameon=True, loc="lower right")
ax.set_title("P1 e P2 - De onde veio a variação do comercial estrito")
plt.tight_layout()
plt.show()

contribuicao = delta[GRUPOS_COMERCIAIS].sum()
print("contribuicao somada no universo (procedentes por mil UCs):")
print(contribuicao.rename(ROTULO_GRUPO).round(2).to_string())

#### Posição em cada ranking:

A tabela reúne as posições nos rankings por grupo e no de recebidas. Uma distribuidora que melhora em tudo aparece com posições baixas em todas as colunas; uma que melhora num grupo só aparece com posições dispersas.

In [ ]:
posicoes = pd.DataFrame({
    "total_procedentes": rk(RECORTE_TOTAL, "procedentes")["posicao"],
    "faturamento": rk("faturamento", "procedentes")["posicao"],
    "pagamento": rk("pagamento", "procedentes")["posicao"],
    "outras_comerciais": rk("outras_comerciais", "procedentes")["posicao"],
    "total_recebidas": rk(RECORTE_TOTAL, "recebidas")["posicao"],
}).sort_values("total_procedentes")
display(posicoes.reset_index())

#### Leitura:

LEITURA_2

## 3. A procedência conta toda a história? (P3)

A procedência é classificada pela própria distribuidora. Uma queda nas procedentes pode vir de o cliente reclamar menos ou de a empresa reconhecer menos. As recebidas não dependem dessa classificação e oferecem um segundo olhar.

In [ ]:
receb = rk(RECORTE_TOTAL, "recebidas")

fig, ax = plt.subplots(figsize=(9, max(5, len(receb) * 0.28)))
cores = [COR_MELHORA if v < 0 else COR_PIORA for v in receb["var_pct"]]
ax.barh(receb.index[::-1], receb["var_pct"][::-1], color=cores[::-1])
ax.axvline(0, color="#37474f", linewidth=1)
ax.set_xlabel(f"Variação das recebidas por mil UCs, {rotulo(INI)} a {rotulo(FIM)} (%)")
ax.tick_params(axis="y", labelsize=8)
ax.set_title("P3 - Comercial estrito: variação das recebidas por mil UCs")
plt.tight_layout()
plt.show()

universo_ini = univ[(univ["recorte"] == RECORTE_TOTAL) & (univ["fim_janela"] == INI)]
universo_fim = univ[(univ["recorte"] == RECORTE_TOTAL) & (univ["fim_janela"] == FIM)]
for nome, col in [("recebidas", "qtd_recebidas"), ("procedentes", "qtd_procedentes")]:
    a, b = universo_ini[col].sum(), universo_fim[col].sum()
    print(f"{nome:<12} universo: {a:>12,.0f} -> {b:>12,.0f} ({(b / a - 1) * 100:+.1f}%)")

#### Posição em procedentes contra posição em recebidas:

Pontos sobre a diagonal têm a mesma posição nos dois rankings. Pontos abaixo da diagonal estão mais bem colocados em procedentes do que em recebidas: reduziram o que reconhecem mais do que aquilo de que o cliente reclama.

In [ ]:
pos = pd.DataFrame({"procedentes": rk(RECORTE_TOTAL, "procedentes")["posicao"],
                    "recebidas": rk(RECORTE_TOTAL, "recebidas")["posicao"]})
pos["diferenca"] = pos["recebidas"] - pos["procedentes"]
n = len(pos)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(pos["procedentes"], pos["recebidas"], color=COR_NEUTRA)
ax.plot([1, n], [1, n], color="#37474f", linewidth=0.8, linestyle="--")
for dx, linha in pos[pos["diferenca"].abs() >= 10].iterrows():
    ax.annotate(dx, (linha["procedentes"], linha["recebidas"]), fontsize=8,
                xytext=(4, 2), textcoords="offset points")
ax.set_xlabel("Posição em procedentes (1 = maior redução)")
ax.set_ylabel("Posição em recebidas (1 = maior redução)")
ax.set_title("P3 - Posição em procedentes e em recebidas")
plt.tight_layout()
plt.show()

rho, p = spearmanr(pos["procedentes"], pos["recebidas"])
print(f"correlacao de Spearman entre as posicoes: {rho:.2f}")
display(pos.sort_values("diferenca", ascending=False).head(8).reset_index())

#### Matriz recebidas × taxa de procedência:

O eixo horizontal mostra a variação das recebidas por mil UCs; o vertical, a variação da taxa de procedência em pontos percentuais. O corte é zero: cada quadrante indica a direção de cada distribuidora, e a distância ao eixo mostra a intensidade.

| Quadrante | Recebidas | Taxa | Leitura |
|---|---|---|---|
| Inferior esquerdo | Queda | Queda | Menos reclamações e menos reconhecimento |
| Superior esquerdo | Queda | Alta | Menos reclamações, e as que chegam procedem mais |
| Inferior direito | Alta | Queda | Alerta: o cliente reclama mais e a empresa reconhece menos |
| Superior direito | Alta | Alta | Piora reconhecida |

In [ ]:
def quadrante(x, y, rotulos):
    """Quadrant label from the sign of the two variations."""
    return rotulos[(x >= 0, y >= 0)]


ROTULO_P3 = {(False, False): "recebidas e taxa em queda",
             (False, True): "recebidas em queda, taxa em alta",
             (True, False): "recebidas em alta, taxa em queda",
             (True, True): "recebidas e taxa em alta"}

matriz = pd.DataFrame({
    "var_recebidas_pct": rk(RECORTE_TOTAL, "recebidas")["var_pct"],
    "var_taxa_pp": (valor(RECORTE_TOTAL, FIM, "taxa_procedencia")
                    - valor(RECORTE_TOTAL, INI, "taxa_procedencia")) * 100,
}).dropna()
matriz["quadrante_p3"] = [quadrante(x, y, ROTULO_P3)
                          for x, y in zip(matriz["var_recebidas_pct"], matriz["var_taxa_pp"])]

fig, ax = plt.subplots(figsize=(9, 6.5))
alerta = matriz["quadrante_p3"] == ROTULO_P3[(True, False)]
ax.scatter(matriz.loc[~alerta, "var_recebidas_pct"], matriz.loc[~alerta, "var_taxa_pp"], color=COR_NEUTRA)
ax.scatter(matriz.loc[alerta, "var_recebidas_pct"], matriz.loc[alerta, "var_taxa_pp"], color=COR_PIORA)
for dx, linha in matriz.iterrows():
    ax.annotate(dx, (linha["var_recebidas_pct"], linha["var_taxa_pp"]), fontsize=7,
                xytext=(3, 2), textcoords="offset points")
ax.axvline(0, color="#37474f", linewidth=1)
ax.axhline(0, color="#37474f", linewidth=1)
ax.set_xlabel(f"Variação das recebidas por mil UCs, {rotulo(INI)} a {rotulo(FIM)} (%)")
ax.set_ylabel("Variação da taxa de procedência (pontos percentuais)")
ax.set_title("P3 - Recebidas e taxa de procedência (em vermelho, quadrante de alerta)")
plt.tight_layout()
plt.show()

print(matriz["quadrante_p3"].value_counts().to_string())
display(matriz.sort_values("var_recebidas_pct").round(1).reset_index())

#### Nível 1 × nível 2:

O cliente que discorda do veredito do nível 1, ou cuja reclamação procedente não foi resolvida, recorre à ouvidoria. A matriz compara a variação das recebidas nos dois níveis, em valores por mil UCs, no total do comercial estrito. Por grupo ou por tipologia a comparação não é confiável (seção 6).

| Quadrante | Nível 1 | Nível 2 | Leitura |
|---|---|---|---|
| Inferior esquerdo | Queda | Queda | Melhora consistente |
| Superior esquerdo | Queda | Alta | Possível migração para a ouvidoria ou falta de resolução no nível 1 |
| Inferior direito | Alta | Queda | Mais reclamações no nível 1, resolvidas sem escalar |
| Superior direito | Alta | Alta | Piora |

In [ ]:
ROTULO_N2 = {(False, False): "N1 e N2 em queda",
             (False, True): "N1 em queda, N2 em alta",
             (True, False): "N1 em alta, N2 em queda",
             (True, True): "N1 e N2 em alta"}

def variacao_pct(coluna):
    ini, fim = valor(RECORTE_TOTAL, INI, coluna), valor(RECORTE_TOTAL, FIM, coluna)
    return (fim / ini - 1) * 100


niveis = pd.DataFrame({"var_n1_pct": variacao_pct("recebidas_por_mil"),
                       "var_n2_pct": variacao_pct("recebidas_nivel_2_por_mil")}).loc[total.index].dropna()
niveis["quadrante_n2"] = [quadrante(x, y, ROTULO_N2) for x, y in zip(niveis["var_n1_pct"], niveis["var_n2_pct"])]

fig, ax = plt.subplots(figsize=(9, 6.5))
migracao = niveis["quadrante_n2"] == ROTULO_N2[(False, True)]
ax.scatter(niveis.loc[~migracao, "var_n1_pct"], niveis.loc[~migracao, "var_n2_pct"], color=COR_NEUTRA)
ax.scatter(niveis.loc[migracao, "var_n1_pct"], niveis.loc[migracao, "var_n2_pct"], color=COR_PIORA)
for dx, linha in niveis.iterrows():
    ax.annotate(dx, (linha["var_n1_pct"], linha["var_n2_pct"]), fontsize=7,
                xytext=(3, 2), textcoords="offset points")
ax.axvline(0, color="#37474f", linewidth=1)
ax.axhline(0, color="#37474f", linewidth=1)
ax.set_xlabel("Variação das recebidas no nível 1 por mil UCs (%)")
ax.set_ylabel("Variação das recebidas no nível 2 por mil UCs (%)")
ax.set_title("P3 - Nível 1 e nível 2 (em vermelho, N1 em queda e N2 em alta)")
plt.tight_layout()
plt.show()

print(niveis["quadrante_n2"].value_counts().to_string())

escalada = pd.DataFrame({"escalada_inicio_pct": valor(RECORTE_TOTAL, INI, "taxa_escalada") * 100,
                         "escalada_fim_pct": valor(RECORTE_TOTAL, FIM, "taxa_escalada") * 100}).loc[total.index]
escalada["variacao_pp"] = escalada["escalada_fim_pct"] - escalada["escalada_inicio_pct"]
tot_n1 = [universo_ini["qtd_recebidas_publicada"].sum(), universo_fim["qtd_recebidas_publicada"].sum()]
tot_n2 = [universo_ini["qtd_recebidas_nivel_2"].sum(), universo_fim["qtd_recebidas_nivel_2"].sum()]
print(f"\nescalada no universo: {tot_n2[0] / tot_n1[0] * 100:.1f}% -> {tot_n2[1] / tot_n1[1] * 100:.1f}%")
print(f"nivel 2 no universo: {tot_n2[0]:,.0f} -> {tot_n2[1]:,.0f} ({(tot_n2[1] / tot_n2[0] - 1) * 100:+.1f}%)")
display(escalada.sort_values("variacao_pp", ascending=False).round(1).reset_index())

#### Leitura:

LEITURA_3

## 4. Técnico e comercial andam juntos? (P4)

Se a mesma gestão que melhora o atendimento comercial melhora a rede, as duas variações deveriam andar juntas. A correlação de Spearman (correlação entre as posições, robusta a valores extremos) mede essa associação; o valor-p responde se ela poderia surgir por pareamento ao acaso. Com 32 distribuidoras, só associações moderadas ou fortes são detectáveis.

In [ ]:
tec_com = pd.DataFrame({"var_qualidade_pct": rk("qualidade", "procedentes")["var_pct"],
                        "var_comercial_pct": rk(RECORTE_TOTAL, "procedentes")["var_pct"]}).dropna()

ROTULO_P4 = {(False, False): "melhora nos dois",
             (False, True): "melhora so no tecnico",
             (True, False): "melhora so no comercial",
             (True, True): "piora nos dois"}
tec_com["quadrante_p4"] = [quadrante(x, y, ROTULO_P4)
                           for x, y in zip(tec_com["var_qualidade_pct"], tec_com["var_comercial_pct"])]

rho_p4, p_p4 = spearmanr(tec_com["var_qualidade_pct"], tec_com["var_comercial_pct"])

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.scatter(tec_com["var_qualidade_pct"], tec_com["var_comercial_pct"], color=COR_NEUTRA)
for dx, linha in tec_com.iterrows():
    ax.annotate(dx, (linha["var_qualidade_pct"], linha["var_comercial_pct"]), fontsize=7,
                xytext=(3, 2), textcoords="offset points")
ax.axvline(0, color="#37474f", linewidth=1)
ax.axhline(0, color="#37474f", linewidth=1)
ax.set_xlabel("Variação das procedentes de Qualidade por mil UCs (%)")
ax.set_ylabel("Variação das procedentes do comercial estrito por mil UCs (%)")
ax.set_title(f"P4 - Técnico e comercial (Spearman {rho_p4:.2f}, valor-p {p_p4:.3f})")
plt.tight_layout()
plt.show()

print(f"Spearman: {rho_p4:.2f}   valor-p: {p_p4:.3f}   distribuidoras: {len(tec_com)}")
print(tec_com["quadrante_p4"].value_counts().to_string())

#### Leitura:

LEITURA_4

## 5. O técnico confirma a continuidade? (P5)

As reclamações de Qualidade são quase todas de falta de energia. Se o pipeline estiver correto, a distribuidora que reduziu o DEC-FI e o FEC-FI deveria ter reduzido também essas reclamações: associação positiva confirma, e associação na direção oposta seria sinal de erro de cálculo. As duas medidas usam as mesmas janelas e o mesmo conjunto de distribuidoras.

In [ ]:
conf = pd.DataFrame({
    "posicao_qualidade": rk("qualidade", "procedentes")["posicao"],
    "var_qualidade_pct": rk("qualidade", "procedentes")["var_pct"],
    "posicao_dec_fi": ranking_cont[ranking_cont["indicador"] == "dec_fi"].set_index("sig_agente")["posicao"],
    "var_dec_fi_pct": ranking_cont[ranking_cont["indicador"] == "dec_fi"].set_index("sig_agente")["var_pct"],
    "posicao_fec_fi": ranking_cont[ranking_cont["indicador"] == "fec_fi"].set_index("sig_agente")["posicao"],
    "var_fec_fi_pct": ranking_cont[ranking_cont["indicador"] == "fec_fi"].set_index("sig_agente")["var_pct"],
}).dropna()
conf["diferenca_dec"] = conf["posicao_qualidade"] - conf["posicao_dec_fi"]

rho_dec, p_dec = spearmanr(conf["var_qualidade_pct"], conf["var_dec_fi_pct"])
rho_fec, p_fec = spearmanr(conf["var_qualidade_pct"], conf["var_fec_fi_pct"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, col, nome, rho, p in [(axes[0], "var_dec_fi_pct", "DEC-FI", rho_dec, p_dec),
                              (axes[1], "var_fec_fi_pct", "FEC-FI", rho_fec, p_fec)]:
    ax.scatter(conf[col], conf["var_qualidade_pct"], color=COR_NEUTRA)
    for dx, linha in conf.iterrows():
        ax.annotate(dx, (linha[col], linha["var_qualidade_pct"]), fontsize=7,
                    xytext=(3, 2), textcoords="offset points")
    ax.axvline(0, color="#37474f", linewidth=1)
    ax.axhline(0, color="#37474f", linewidth=1)
    ax.set_xlabel(f"Variação do {nome} (%)")
    ax.set_ylabel("Variação das procedentes de Qualidade (%)")
    ax.set_title(f"{nome}: Spearman {rho:.2f}, valor-p {p:.3f}")
fig.suptitle(f"P5 - Reclamações de Qualidade e continuidade, {rotulo(INI)} a {rotulo(FIM)}")
plt.tight_layout()
plt.show()

print(f"DEC-FI: Spearman {rho_dec:.2f}, valor-p {p_dec:.3f}")
print(f"FEC-FI: Spearman {rho_fec:.2f}, valor-p {p_fec:.3f}")
display(conf.sort_values("posicao_dec_fi").reset_index())

#### Divergências:

As distribuidoras com maior diferença entre a posição em Qualidade e a posição no DEC-FI são as que a leitura precisa explicar. Parte delas está ligada a eventos climáticos na janela inicial: as enchentes do Rio Grande do Sul em maio de 2024 elevam a base de CEEE-D e RGE SUL, e a redução posterior é em boa parte efeito de base.

In [ ]:
display(conf.reindex(conf["diferenca_dec"].abs().sort_values(ascending=False).index)
        .head(8)[["posicao_qualidade", "posicao_dec_fi", "diferenca_dec",
                  "var_qualidade_pct", "var_dec_fi_pct"]].reset_index())

#### Leitura:

LEITURA_5

## 6. Robustez

Quatro verificações mostram quanto as conclusões dependem de decisões de tratamento ou de limitações do dado.

#### Sensibilidade à imputação:

Seis meses do comercial estrito foram imputados na Silver. A tabela mostra as distribuidoras cuja posição muda quando o ranking é calculado com os valores publicados.

In [ ]:
sens = rk(RECORTE_TOTAL, "procedentes")
mudam = sens[sens["variacao_posicao"] != 0][["posicao", "posicao_sem_imputacao", "variacao_posicao",
                                              "var_pct", "var_pct_sem_imputacao"]]
print(f"distribuidoras com posicao alterada pela imputacao: {len(mudam)} de {len(sens)}")
display(mudam.reset_index())

#### Volume baixo:

Nos grupos de volume baixo por distribuidora, poucas dezenas de reclamações movem a variação percentual. A tabela mostra as posições extremas de Pagamento e Outras comerciais ao lado do volume absoluto.

In [ ]:
for grupo in ["pagamento", "outras_comerciais"]:
    tabela = rk(grupo, "procedentes")[["posicao", "var_pct", "volume_inicio", "volume_fim"]]
    print(f"{ROTULO_GRUPO[grupo]}: volume final mediano de {tabela['volume_fim'].median():,.0f} procedentes")
    display(pd.concat([tabela.head(3), tabela.tail(3)]).reset_index())

#### Inconsistência entre os níveis 1 e 2:

Por tipologia, há distribuidoras em que o nível 2 recebe mais reclamações do que o nível 1 inteiro, todos os meses da série. A tabela lista esses casos na última janela. É o motivo de a comparação entre níveis se restringir ao total do comercial estrito, onde o nível 2 é menor que o nível 1 em todas as distribuidoras.

In [ ]:
por_tipologia = (fato.filter(F.col("ano_mes").isin(meses_fim))
    .join(universo_spark, "num_cnpj")
    .join(tipologia.filter("ind_comercial_estrito").select("cod_tipologia", "descricao"), "cod_tipologia")
    .groupBy("num_cnpj", "cod_tipologia", "descricao")
    .agg(F.sum(F.when(F.col("nivel") == 1, F.col("qtd_recebidas_publicada")).otherwise(0)).alias("recebidas_n1"),
         F.sum(F.when(F.col("nivel") == 2, F.col("qtd_recebidas_publicada")).otherwise(0)).alias("recebidas_n2"))
    .filter(F.col("recebidas_n2") > F.col("recebidas_n1"))
    .toPandas().merge(nomes, on="num_cnpj"))

print(f"combinacoes distribuidora x tipologia com N2 acima de N1: {len(por_tipologia)}")
print(f"distribuidoras envolvidas: {por_tipologia['sig_agente'].nunique()}")
display(por_tipologia.sort_values("recebidas_n2", ascending=False)
        [["sig_agente", "cod_tipologia", "descricao", "recebidas_n1", "recebidas_n2"]].head(15))

#### Exclusões:

A CELESC está fora de todos os rankings. A decisão e a evidência estão no `02_silver_complaints`, seção 11.

In [ ]:
display(spark.table(f"{SILVER}.exclusao_ranking_manifestacao").toPandas().merge(nomes, on="num_cnpj"))

#### Leitura:

LEITURA_6

## 7. Discussão geral

O quadro reúne, por distribuidora, as respostas às cinco perguntas: posição no comercial estrito por procedentes e por recebidas, posição em cada grupo, quadrante da P3 nas duas matrizes, e posição em Qualidade e no DEC-FI.

In [ ]:
quadro = (posicoes
    .join(matriz["quadrante_p3"])
    .join(niveis["quadrante_n2"])
    .join(conf[["posicao_qualidade", "posicao_dec_fi"]])
    .sort_values("total_procedentes"))
display(quadro.reset_index())

DISCUSSAO_GERAL

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Faixa de estabilidade nas matrizes | Corte em zero, sem faixa neutra | Definir a faixa com base estatística, por exemplo a dispersão mensal da própria distribuidora |
| Inferência sobre a série mensal | Resultados descritivos | Gráfico de controle tipo u, Mann-Kendall e bootstrap de posição sobre a série mensal |
| Eventos climáticos na base | Efeito de base apontado na P5 | Estudo em notebook próprio, com ranking que neutraliza os meses de evento, após a conclusão do trabalho |
| Inconsistência entre níveis | Comparação restrita ao total | Apurar a causa junto às distribuidoras ou à ANEEL |

## Autoavaliação desta etapa

AUTOAVALIACAO